# BANKSY per-group domain analysis

Runs a per-group (one injury type × timepoint at a time) BANKSY + Harmony
spatial domain analysis, then pools domain profiles across all groups to
identify **meta-domains** — tissue regions that recur consistently across
conditions and timepoints.

## Workflow

1. **k_geom QC** — compute median k-NN distances per group to guide `k_geom` selection
2. **Per-group BANKSY + Harmony** — for each `GROUP_VALUE` in `inj_type_per_day`:
   - Subset to the group, exclude blacklisted batches
   - Normalize (CPM) + log1p
   - Stagger slide coordinates by batch for visualization
   - Run BANKSY (λ=0.8) → PCA (25 components) → Harmony batch correction
   - Cluster with two methods at multiple resolutions:
     - **Method A** — BANKSY-style Leiden (`run_Leiden_partition`)
     - **Method B** — Scanpy-style Leiden (`sc.pp.neighbors` on `X_pca_harmony`)
   - Save domain expression profiles (mean log-normalized expression per domain)
   - Save the integrated `.h5ad` with `counts` + `lognorm` layers and raw counts
3. **Meta-domain discovery** — load all per-group domain profiles, z-score across
   genes, compute pairwise domain-domain correlation, hierarchically cluster
   domains into `t` meta-domains that are shared across conditions

## Usage

Run the pipeline cell once per group by changing:
```python
GROUP_VALUE = "inj_1_d5"   # which group to process
k_geom = 8                  # spatial neighbourhood size (from QC cell above)
```

**Input:** merged `.h5ad` with `obs['inj_type_per_day']`, `obs['batch']`, `obs['x']`, `obs['y']`

**Outputs per group** (in `OUT_DIR/<GROUP_VALUE>/`):
- `adata_group_banksy_harmony.h5ad`
- `domain_profiles__BANKSY__<cluster_key>.csv`
- `domain_profiles__SCANPY__<cluster_key>.csv`
- `UMAP_domains_grid__<GROUP_VALUE>.png`
- `SPATIAL__HARMONY__<GROUP_VALUE>.png`
- `SPATIAL__BANKSY__<GROUP_VALUE>.png`

**Output (meta-domain step):** `meta_domains_mapping.csv`

## 1 · k_geom QC

Compute median k-NN distances per group to guide `k_geom` selection.

In [ ]:
adata = sc.read_h5ad("/path/to/your/merged_dataset.h5ad")

from sklearn.neighbors import NearestNeighbors
import numpy as np
import pandas as pd

# if coord_xy does not exist, create it from x/y
if "coord_xy" not in adata.obsm:
    adata.obsm["coord_xy"] = np.vstack([adata.obs["x"].to_numpy(),
                                        adata.obs["y"].to_numpy()]).T

coords = adata.obsm["coord_xy"]

results = []
for group, idx in adata.obs.groupby("inj_type_per_day").indices.items():
    sub_coords = coords[idx]

    if sub_coords.shape[0] < 12:
        continue

    nbrs = NearestNeighbors(n_neighbors=12)
    nbrs.fit(sub_coords)
    dist, _ = nbrs.kneighbors(sub_coords)
    dist = dist[:, 1:]  # remove self

    results.append({
        "group": group,
        "median_k7": np.median(dist[:, 6]),
        "median_k8": np.median(dist[:, 7]),
        "median_k10": np.median(dist[:, 9]),
        "median_k11": np.median(dist[:,10]),
        "n_cells": sub_coords.shape[0]
    })

df = pd.DataFrame(results).sort_values("median_k8")
print(df)

## 2 · Per-batch k-NN distance distribution

Visualize the spatial density per batch.

In [ ]:
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt
import numpy as np

k_test = 8   # same as your k_geom

coords = adata.obsm["coord_xy"]

nbrs = NearestNeighbors(n_neighbors=k_test).fit(coords)
distances, _ = nbrs.kneighbors(coords)

# distance to kth neighbor
k_dist = distances[:, -1]

adata.obs["k_dist"] = k_dist

plt.figure(figsize=(10,5))

for b in sorted(adata.obs["batch"].unique()):
    vals = adata.obs.loc[adata.obs["batch"] == b, "k_dist"]
    plt.hist(vals, bins=100, alpha=0.4, label=b)

plt.xlabel(f"Distance to {k_test}th neighbor")
plt.ylabel("Cells")
plt.title("Spatial neighbor distance distribution")
plt.legend(fontsize=6)
plt.show()

## 3 · Per-group pipeline

Edit `GROUP_VALUE` and `k_geom` then run this cell for each group.
All other parameters (`resolutions`, `pca_dim`, `lambda_list`) are fixed.

In [ ]:
# ============================================================
# Per-group (inj_type_per_day) BANKSY → Harmony → TWO Leiden methods
# + Domain expression profiles
# + SAVE with: layers["counts"], layers["lognorm"], and .raw (counts)
#
# NEW:
#   - Save GRID PNGs of "spatial" (staggered) projections for ALL clusterings
#   - PNGs tuned to be lightweight:
#       * moderate DPI
#       * small marker size
#       * rasterized scatter (helps especially if you ever save vector formats)
#
# You only change:
#   - GROUP_VALUE (inj_type_per_day)
#   - k_geom
#
# Methods:
#   A) BANKSY-style Leiden (run_Leiden_partition) AFTER Harmony
#      (we overwrite reduced_pc_* with harmony PCs, notebook-style)
#   B) Scanpy-style Leiden (sc.pp.neighbors on X_pca_harmony) AFTER Harmony
#
# Resolutions for BOTH:
#   [0.5, 0.3, 0.8, 1.0, 1.2]
#
# Input:
#   all_slides/sectionsOK_rawdata.h5ad
#
# Output:
#   all_slides/banksy2/per_group/<GROUP_VALUE>/
#     - adata_group_banksy_harmony.h5ad
#     - stagger_check.png
#     - domain_profiles__BANKSY__<params_name>.csv
#     - domain_sizes__BANKSY__<params_name>.csv
#     - domain_profiles__SCANPY__leiden_<r>.csv
#     - domain_sizes__SCANPY__leiden_<r>.csv
# ============================================================

import os
import math
import numpy as np
import pandas as pd
import scanpy as sc
import scanpy.external as sce
import scipy.sparse as sp
import matplotlib.pyplot as plt

from banksy.initialize_banksy import initialize_banksy
from banksy.embed_banksy import generate_banksy_matrix
from banksy_utils.umap_pca import pca_umap
from banksy.cluster_methods import run_Leiden_partition


# -------------------------
# USER SETTINGS (change only these)
# -------------------------
ALL_SLIDES_ROOT = "/path/to/your/data/"
IN_PATH = os.path.join(ALL_SLIDES_ROOT, "merged_dataset.h5ad")

GROUP_KEY = "inj_type_per_day"
GROUP_VALUE = "inj_1_d5"   # e.g. inj_1_d5, inj_2_d7, uninjured     # <-- CHANGE THIS ONLY
k_geom = 8                   # <-- CHANGE THIS ONLY (8 or 10)


# -------------------------
# Fixed settings
# -------------------------
batch_key = "batch"
coord_x, coord_y = "x", "y"

nbr_weight_decay = "scaled_gaussian"
max_m = 1
lambda_list = [0.8]
pca_dim = 25
resolutions = [0.5, 0.3, 0.8, 1.0, 1.2]
seed = 0

scanpy_n_neighbors = 30
umap_min_dist = 0.001
spacing_factor = 2.0

OUT_DIR = os.path.join(ALL_SLIDES_ROOT, "banksy_domains", "per_group", str(GROUP_VALUE)))
os.makedirs(OUT_DIR, exist_ok=True)
OUT_H5AD = os.path.join(OUT_DIR, "adata_group_banksy_harmony.h5ad")


# -------------------------
# BLACKLIST (same as yours)
# -------------------------
exclude_batches = [
    # List batch IDs to exclude (e.g. sections with quality issues,
    # insufficient cell counts, or imaging artefacts).
    # Example: "inj_1_d9_4", "uninjured_6"
]
exclude_batches = sorted(set([str(x).strip() for x in exclude_batches]))


# -------------------------
# Helpers
# -------------------------
def _get_expr_matrix(ad):
    """Expression matrix for profiles: prefer lognorm layer."""
    X = ad.layers["lognorm"] if "lognorm" in ad.layers else ad.X
    if sp.issparse(X):
        X = X.toarray()
    return X

def _write_profiles(ad, cluster_key, prefix, out_dir):
    """Mean expression per domain + sizes."""
    X = _get_expr_matrix(ad)
    genes = ad.var_names.to_numpy()
    groups = ad.obs[cluster_key].astype(str).to_numpy()

    df = pd.DataFrame(X, columns=genes)
    df["domain"] = groups
    profiles = df.groupby("domain", sort=True).mean()
    sizes = pd.Series(groups).value_counts().sort_index()

    profiles.to_csv(os.path.join(out_dir, f"domain_profiles__{prefix}__{cluster_key}.csv"))
    sizes.to_csv(os.path.join(out_dir, f"domain_sizes__{prefix}__{cluster_key}.csv"), header=["n_cells"])

def save_stagger_grid_png(
    ad,
    cluster_keys,
    out_path,
    title,
    x_key="x_stag",
    y_key="y_stag",
    ncols=3,
    point_size=0.6,
    dpi=180,
    max_legend=25,
):
    """
    Save a GRID of staggered spatial projections (PNG).
    Notes on file size:
      - PNG is raster already; main levers are figure size + dpi + point_size.
      - rasterized=True is added (useful if you ever switch to PDF/SVG too).
    """
    # Filter to existing keys only (avoid hard crashes mid-run)
    cluster_keys = [k for k in cluster_keys if k in ad.obs.columns]
    if len(cluster_keys) == 0:
        raise ValueError("No cluster keys found in ad.obs for grid plotting.")

    x = ad.obs[x_key].to_numpy()
    y = ad.obs[y_key].to_numpy()

    n = len(cluster_keys)
    nrows = math.ceil(n / ncols)

    # Wide + short works well for your stagger layout
    fig_w = 6.2 * ncols
    fig_h = 3.2 * nrows
    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(fig_w, fig_h), squeeze=False)

    for i, key in enumerate(cluster_keys):
        ax = axes[i // ncols][i % ncols]

        lab = ad.obs[key].astype(str)
        cats = pd.Categorical(lab)
        codes = cats.codes

        sca = ax.scatter(
            x, y,
            c=codes,
            s=point_size,
            cmap="tab20",
            linewidths=0,
            rasterized=True,   # keeps light if you later save vector formats; harmless for PNG
        )
        ax.invert_yaxis()
        ax.set_title(key, fontsize=10)
        ax.set_xlabel(x_key)
        ax.set_ylabel(y_key)

        # Optional small legend (capped)
        uniq = list(cats.categories)
        if len(uniq) <= max_legend:
            handles = []
            for j, name in enumerate(uniq):
                handles.append(
                    plt.Line2D(
                        [0], [0],
                        marker="o",
                        linestyle="",
                        markersize=4,
                        markerfacecolor=sca.cmap(sca.norm(j)),
                        markeredgewidth=0,
                        label=name,
                    )
                )
            ax.legend(handles=handles, title="cluster", fontsize=7, title_fontsize=8,
                      loc="upper left", bbox_to_anchor=(1.02, 1.0), borderaxespad=0.0)

    # Turn off unused axes
    for j in range(n, nrows * ncols):
        axes[j // ncols][j % ncols].axis("off")

    fig.suptitle(title, fontsize=14)
    fig.tight_layout(rect=[0, 0, 1, 0.95])

    fig.savefig(out_path, dpi=dpi)
    plt.close(fig)

def ensure_spatial_from_stagger(ad, x_key="x_stag", y_key="y_stag", obsm_key="spatial"):
    """Create ad.obsm['spatial'] from stagger coordinates."""
    import numpy as np
    ad.obsm[obsm_key] = np.c_[
        ad.obs[x_key].to_numpy(),
        ad.obs[y_key].to_numpy()
    ].astype(np.float32)


def save_umap_grid(ad, color_keys, out_png, ncols=3, dpi=300):
    """Save a grid of UMAP plots."""
    import matplotlib.pyplot as plt
    import scanpy as sc

    color_keys = [k for k in color_keys if k in ad.obs.columns]

    sc.pl.umap(
        ad,
        color=color_keys,
        ncols=ncols,
        show=False
    )

    plt.savefig(out_png, dpi=dpi, bbox_inches="tight")
    plt.close()

def save_spatial_grid_rows(ad, color_keys, out_png, spot_size=12, dpi=300, hspace=0.12):
    color_keys = [k for k in color_keys if k in ad.obs.columns]
    if len(color_keys) == 0:
        raise ValueError("No requested keys found in ad.obs")

    if "spatial" not in ad.obsm_keys():
        ensure_spatial_from_stagger(ad)

    n = len(color_keys)
    fig, axes = plt.subplots(n, 1, figsize=(20, 3.6 * n))
    if n == 1:
        axes = [axes]

    for i, key in enumerate(color_keys):
        sc.pl.spatial(
            ad,
            color=key,
            basis="spatial",
            img_key=None,
            spot_size=spot_size,
            ax=axes[i],
            show=False,
            legend_loc="right margin",
        )
        axes[i].set_xticks([])
        axes[i].set_yticks([])

    plt.subplots_adjust(hspace=hspace)
    plt.savefig(out_png, dpi=dpi, bbox_inches="tight")
    plt.close(fig)

# -------------------------
# 1) Load + subset
# -------------------------
adata = sc.read_h5ad(IN_PATH)

for col in [batch_key, GROUP_KEY, coord_x, coord_y]:
    if col not in adata.obs.columns:
        raise KeyError(f"Missing adata.obs['{col}']")

adata = adata[~adata.obs[batch_key].astype(str).isin(exclude_batches)].copy()
adata = adata[adata.obs[GROUP_KEY].astype(str) == str(GROUP_VALUE)].copy()

print(f"[{GROUP_VALUE}] n_obs={adata.n_obs}, n_vars={adata.n_vars}, batches={adata.obs[batch_key].nunique()}")
if adata.n_obs < 500:
    raise ValueError("Too few cells/spots for this group after filtering.")


# -------------------------
# 2) Make layers counts + lognorm (and keep a counts .raw later)
# -------------------------
if not sp.issparse(adata.X):
    adata.X = sp.csr_matrix(adata.X)

# store counts
adata.layers["counts"] = adata.X.copy()

# normalize + log1p
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.layers["lognorm"] = adata.X.copy()

# BANKSY uses lognorm
adata.X = adata.layers["lognorm"].copy()
if not sp.isspmatrix_csr(adata.X):
    adata.X = adata.X.tocsr()
adata.X = adata.X.astype(np.float32)


# -------------------------
# 3) Stagger coords by batch
# -------------------------
coords = adata.obs[[coord_x, coord_y, batch_key]].copy()
coords[coord_x] = coords[coord_x].astype(float)
coords[coord_y] = coords[coord_y].astype(float)

coords["x0"] = coords.groupby(batch_key)[coord_x].transform(lambda v: v - v.min())
coords["y0"] = coords.groupby(batch_key)[coord_y].transform(lambda v: v - v.min())

global_max_x = coords["x0"].max()
spacing = global_max_x * spacing_factor
batch_codes = pd.Categorical(coords[batch_key]).codes

coords["x_stag"] = coords["x0"] + batch_codes * spacing
coords["y_stag"] = coords["y0"]

adata.obs["x_stag"] = coords["x_stag"].to_numpy()
adata.obs["y_stag"] = coords["y_stag"].to_numpy()

coord_keys = ("x_stag", "y_stag", "coord_xy")
adata.obsm["coord_xy"] = np.vstack([
    adata.obs["x_stag"].to_numpy(),
    adata.obs["y_stag"].to_numpy()
]).T.astype(np.float32)

# quick stagger plot
plt.figure(figsize=(18, 5))
plt.scatter(
    adata.obs["x_stag"], adata.obs["y_stag"], s=1,
    c=pd.Categorical(adata.obs[batch_key]).codes, cmap="tab20",
    linewidths=0, rasterized=True
)
plt.gca().invert_yaxis()
plt.title(f"Staggered sections: {GROUP_VALUE}")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "stagger_check.png"), dpi=180)
plt.close()


# -------------------------
# 4) BANKSY + PCA
# -------------------------
banksy_dict = initialize_banksy(
    adata,
    coord_keys,
    k_geom,
    nbr_weight_decay=nbr_weight_decay,
    max_m=max_m,
    plt_edge_hist=False,
    plt_nbr_weights=False,
    plt_agf_angles=False,
    plt_theta=False,
)

banksy_dict, _ = generate_banksy_matrix(adata, banksy_dict, lambda_list, max_m)
pca_umap(banksy_dict, pca_dims=[pca_dim], add_umap=False, plt_remaining_var=False)

lam = float(lambda_list[0])
adata_joint = banksy_dict[nbr_weight_decay][lam]["adata"].copy()
pc_key = f"reduced_pc_{pca_dim}"


# -------------------------
# 5) Harmony on BANKSY PCs
# -------------------------
sce.pp.harmony_integrate(adata_joint, key=batch_key, basis=pc_key, max_iter_harmony=50)

# overwrite so downstream uses harmony-corrected PCs
adata_joint.obsm[pc_key] = adata_joint.obsm["X_pca_harmony"].copy()
adata_joint.obsm[f"{pc_key}_harmony"] = adata_joint.obsm["X_pca_harmony"].copy()


# -------------------------
# 6A) Leiden Method A (BANKSY-style) AFTER Harmony
# -------------------------
banksy_results_df, _ = run_Leiden_partition(
    banksy_dict,
    list(resolutions),
    num_nn=50,
    num_iterations=-1,
    partition_seed=seed,
    match_labels=False,
)

banksy_cluster_cols = []
for params_name in banksy_results_df.index:
    labels = banksy_results_df.loc[params_name, "labels"].dense.astype(str)
    col = f"banksy_{params_name}"
    adata_joint.obs[col] = pd.Categorical(labels)
    banksy_cluster_cols.append(col)
    _write_profiles(adata_joint, col, prefix="BANKSY", out_dir=OUT_DIR)


# -------------------------
# 6B) Leiden Method B (Scanpy-style) AFTER Harmony
# -------------------------
sc.pp.neighbors(adata_joint, use_rep="X_pca_harmony", n_neighbors=scanpy_n_neighbors)
sc.tl.umap(adata_joint, min_dist=umap_min_dist)
adata_joint.obsm["X_umap_harmony"] = adata_joint.obsm["X_umap"].copy()

scanpy_cluster_cols = []
for r in resolutions:
    col = f"harmony_leiden_{r:g}"
    sc.tl.leiden(adata_joint, resolution=float(r), key_added=col)
    scanpy_cluster_cols.append(col)
    _write_profiles(adata_joint, col, prefix="SCANPY", out_dir=OUT_DIR)

# -------------------------
# 7) Build a GENE-ONLY object for saving counts/lognorm/raw
# -------------------------
# Keep only genes that exist in the original input (drops *_nbr_* features)
genes_keep = adata.var_names.intersection(adata_joint.var_names)

# create a save object with same obs, but only real genes
adata_save = adata_joint[:, genes_keep].copy()

# align original adata to adata_save (safe label-based slicing)
adata_aligned = adata[adata_save.obs_names, adata_save.var_names]

# attach layers from original adata
adata_save.layers["counts"] = adata_aligned.layers["counts"].copy()
adata_save.layers["lognorm"] = adata_aligned.layers["lognorm"].copy()

# set X = lognorm
adata_save.X = adata_save.layers["lognorm"].copy()
if sp.issparse(adata_save.X):
    adata_save.X = adata_save.X.tocsr()

# lightweight raw = counts (raw var can differ from X in general, but here it matches)
adata_save.raw = sc.AnnData(
    X=adata_save.layers["counts"],
    obs=adata_save.obs.copy(),
    var=adata_save.var.copy(),
)

print("Saved object shape:", adata_save.shape)
print("Layers:", list(adata_save.layers.keys()))
print("Raw present:", adata_save.raw is not None)

# -------------------------
# 8) Save
# -------------------------
adata_save.write(OUT_H5AD)
print("Saved:", OUT_H5AD)
print("Outputs in:", OUT_DIR)

# -------------------------
# 9) SAVE PLOTS (UMAP grid + Spatial grid)
# -------------------------
ad_plot = adata_save  # <-- use the in-memory object you are saving

# Ensure UMAP is present under X_umap (sometimes you stored X_umap_harmony)
if "X_umap" not in ad_plot.obsm_keys() and "X_umap_harmony" in ad_plot.obsm_keys():
    ad_plot.obsm["X_umap"] = ad_plot.obsm["X_umap_harmony"].copy()

# Make sure spatial coords exist for plotting
ensure_spatial_from_stagger(ad_plot, obsm_key="spatial")

# Choose cluster keys (keep your exact order)
umap_keys = [
    "harmony_leiden_0.3",
    "harmony_leiden_0.5",
    "harmony_leiden_0.8",
    "harmony_leiden_1",
    "harmony_leiden_1.2",
    "banksy_scaled_gaussian_pc25_nc0.80_r0.30",
    "banksy_scaled_gaussian_pc25_nc0.80_r0.50",
    "banksy_scaled_gaussian_pc25_nc0.80_r0.80",
    "banksy_scaled_gaussian_pc25_nc0.80_r1.00",
    "banksy_scaled_gaussian_pc25_nc0.80_r1.20",
]

# 1) UMAP grid (all panels together)
out_umap = os.path.join(OUT_DIR, f"UMAP_domains_grid__{GROUP_VALUE}.png")
save_umap_grid(ad_plot, umap_keys, out_umap, ncols=3, dpi=300)

# Choose plotting object (use adata_save if you created it; else adata_joint)
ad_plot = adata_save  # or adata_joint

# Collect keys
harmony_keys = [k for k in ad_plot.obs.columns if k.startswith("harmony_leiden_")]
banksy_keys  = [k for k in ad_plot.obs.columns if k.startswith("banksy_")]

# Optional: sort nicely by resolution (numbers at end)
def _sort_by_last_number(keys):
    def last_num(s):
        import re
        nums = re.findall(r"(\d+\.?\d*)", s)
        return float(nums[-1]) if nums else 1e9
    return sorted(keys, key=last_num)

harmony_keys = _sort_by_last_number(harmony_keys)
banksy_keys  = _sort_by_last_number(banksy_keys)

# Save Harmony spatial PNG (one figure)
save_spatial_grid_rows(
    ad_plot,
    harmony_keys,
    os.path.join(OUT_DIR, f"SPATIAL__HARMONY__{GROUP_VALUE}.png"),
    spot_size=12,
    dpi=300,
    hspace=0.10
)

# Save BANKSY spatial PNG (one figure)
save_spatial_grid_rows(
    ad_plot,
    banksy_keys,
    os.path.join(OUT_DIR, f"SPATIAL__BANKSY__{GROUP_VALUE}.png"),
    spot_size=12,
    dpi=300,
    hspace=0.10
)

print("Saved:")
print("  ", os.path.join(OUT_DIR, f"SPATIAL__HARMONY__{GROUP_VALUE}.png"))
print("  ", os.path.join(OUT_DIR, f"SPATIAL__BANKSY__{GROUP_VALUE}.png"))
print("Saved plots:")
print("  ", out_umap)
print("  ", out_spatial)

## 4 · Meta-domain discovery

Load all per-group domain profiles, compute cross-condition domain-domain
correlation, and cut the dendrogram into `t` meta-domains.

Edit `t` to change the number of meta-domains.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.cluster.hierarchy import linkage, fcluster, leaves_list
from scipy.spatial.distance import squareform

ROOT = "/path/to/your/banksy_domains/per_group/"
pattern = "domain_profiles__BANKSY__banksy_scaled_gaussian_pc25_nc0.80_r0.50.csv"

# -----------------------------
# 1. Load all domain profiles
# -----------------------------
profiles = {}

for group in os.listdir(ROOT):
    f = os.path.join(ROOT, group, pattern)
    if os.path.exists(f):
        df = pd.read_csv(f, index_col=0)

        # make domain names unique across groups
        df.index = [f"{group}__{d}" for d in df.index]
        profiles[group] = df

ALL = pd.concat(profiles.values(), axis=0)

print("Total domains:", ALL.shape[0])
print("Genes:", ALL.shape[1])

# -----------------------------
# 2. Z-score genes across domains
# -----------------------------
X = ALL.values.astype(float)
X = (X - X.mean(axis=0)) / (X.std(axis=0) + 1e-8)

ALL_Z = pd.DataFrame(X, index=ALL.index, columns=ALL.columns)

# -----------------------------
# 3. Domain-domain correlation
# -----------------------------
corr = np.corrcoef(ALL_Z.values)   # row-vs-row correlation
corr = pd.DataFrame(corr, index=ALL_Z.index, columns=ALL_Z.index)

print("Correlation matrix shape:", corr.shape)

# -----------------------------
# 4. Build distance matrix
# -----------------------------
dist = 1 - corr.values

# numerical cleanup
dist = np.clip(dist, 0, 2)
np.fill_diagonal(dist, 0)

# linkage needs condensed distance matrix
dist_condensed = squareform(dist, checks=False)
Z = linkage(dist_condensed, method="average")

# -----------------------------
# 5. Cut dendrogram into meta-domains
# -----------------------------
t = 9   # number of meta-domains
meta = fcluster(Z, t=t, criterion="maxclust")

meta_df = pd.DataFrame({"meta_domain": meta}, index=corr.index.astype(str))
meta_df["condition"] = meta_df.index.str.split("__").str[0]
meta_df["domain"] = meta_df.index.str.split("__").str[1]

meta_df.to_csv(os.path.join(ROOT, "meta_domains_mapping.csv"))

conservation = (
    meta_df.groupby("meta_domain")["condition"]
    .nunique()
    .sort_values(ascending=False)
)

print("Number of meta-domains:", meta_df["meta_domain"].nunique())
print(conservation.head(10))

# -----------------------------
# 6A. Order by dendrogram leaves
# -----------------------------
order = leaves_list(Z)
corr_sorted = corr.iloc[order, order]
meta_sorted = meta_df.iloc[order]

# -----------------------------
# 6B. Optional cleaner order:
# sort by meta-domain, then condition
# -----------------------------
meta_df_for_sort = meta_df.copy()
meta_df_for_sort["orig_idx"] = np.arange(len(meta_df_for_sort))

sort_idx = (
    meta_df_for_sort
    .sort_values(["meta_domain", "condition", "domain"])
    ["orig_idx"]
    .values
)

corr_meta_sorted = corr.iloc[sort_idx, sort_idx]
meta_sorted2 = meta_df.iloc[sort_idx]

# -----------------------------
# 7. Plot function
# -----------------------------
def plot_corr(corr_mat, annot_df, title):
    plt.figure(figsize=(14, 14))
    im = plt.imshow(corr_mat, cmap="RdBu_r", vmin=-1, vmax=1)
    plt.colorbar(im, label="domain correlation")

    # draw boundaries between meta-domains
    meta_vals = annot_df["meta_domain"].values
    boundaries = np.where(meta_vals[:-1] != meta_vals[1:])[0]

    plt.title(title)
    plt.xlabel("domains")
    plt.ylabel("domains")
    plt.xticks([])
    plt.yticks([])
    plt.tight_layout()
    plt.show()

# dendrogram order
plot_corr(
    corr_sorted,
    meta_sorted,
    "Spatial domain similarity across conditions (dendrogram order)"
)
